# Char-level GPT on Tiny Shakespeare

Trains the decoder-only Transformer defined in [`gpt.py`](./gpt.py) as a
character-level language model on the Tiny Shakespeare corpus
(`data/input.txt`).

Pipeline:
1. Load the raw text and build a character-level vocabulary.
2. Encode the corpus and split into train/val.
3. Sample random `block_size`-length chunks as training batches.
4. Train `GPT` with AdamW, tracking train/val loss.
5. Sample new Shakespeare-ish text from the trained model.

Run cells top to bottom. This notebook must live in the same directory as
`gpt.py` (it imports `GPT` directly from it).

## 1. Imports & device

In [1]:
import math
import time

import torch
import torch.nn.functional as F

from gpt import GPT

torch.manual_seed(1337)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using device: {device}")

using device: cpu


## 2. Load the data

In [2]:
with open("data/input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(f"dataset length (chars): {len(text):,}")
print("---- sample ----")
print(text[:300])

dataset length (chars): 1,115,394
---- sample ----
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


## 3. Character-level vocabulary

Every unique character in the corpus becomes one token. This keeps the
tokenizer trivial (no BPE) at the cost of longer sequences per sentence.

In [3]:
chars = sorted(set(text))
vocab_size = len(chars)
print(f"vocab size: {vocab_size}")
print("".join(chars))

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s: str) -> list[int]:
    return [stoi[c] for c in s]

def decode(ids: list[int]) -> str:
    return "".join(itos[i] for i in ids)

assert decode(encode("hello there")) == "hello there"

vocab size: 65

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


## 4. Train / validation split

In [5]:
data = torch.tensor(encode(text), dtype=torch.long)

split = int(0.9 * len(data))
train_data = data[:split]
val_data = data[split:]

print(f"train tokens: {len(train_data):,}")
print(f"val tokens:   {len(val_data):,}")

train tokens: 1,003,854
val tokens:   111,540


## 5. Batching

Each training example is a random `block_size`-length window of the corpus;
the target sequence is the same window shifted one character to the right
(next-character prediction).

In [6]:
def get_batch(split_name: str, block_size: int, batch_size: int):
    source = train_data if split_name == "train" else val_data
    ix = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i : i + block_size] for i in ix])
    y = torch.stack([source[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

## 6. Hyperparameters & model

In [7]:
# Kept small so a CPU can train this in a reasonable time. Bump these up
# (and max_iters) if you have a GPU.
block_size = 128     # context length
batch_size = 64
d_model = 128
n_heads = 4
n_layers = 4
d_ff = 4 * d_model
dropout = 0.1

max_iters = 3000
eval_interval = 250
eval_iters = 50
learning_rate = 3e-4

model = GPT(
    vocab_size=vocab_size,
    d_model=d_model,
    n_heads=n_heads,
    n_layers=n_layers,
    d_ff=d_ff,
    max_len=block_size,
    dropout=dropout,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"model parameters: {n_params:,}")

model parameters: 810,049


## 7. Optimizer & loss estimation

In [8]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split_name in ("train", "val"):
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split_name, block_size, batch_size)
            logits = model(xb)
            b, t, c = logits.shape
            loss = F.cross_entropy(logits.view(b * t, c), yb.view(b * t))
            losses[k] = loss.item()
        out[split_name] = losses.mean().item()
    model.train()
    return out

## 8. Training loop

In [9]:
model.train()
start = time.time()

for it in range(max_iters):
    if it % eval_interval == 0 or it == max_iters - 1:
        losses = estimate_loss()
        elapsed = time.time() - start
        print(
            f"step {it:5d} | train loss {losses['train']:.4f} | "
            f"val loss {losses['val']:.4f} | {elapsed:.1f}s"
        )

    xb, yb = get_batch("train", block_size, batch_size)
    logits = model(xb)
    b, t, c = logits.shape
    loss = F.cross_entropy(logits.view(b * t, c), yb.view(b * t))

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"done in {time.time() - start:.1f}s")

step     0 | train loss 4.3419 | val loss 4.3496 | 31.9s
step   250 | train loss 2.5219 | val loss 2.5304 | 280.8s
step   500 | train loss 2.4548 | val loss 2.4605 | 512.9s
step   750 | train loss 2.4156 | val loss 2.4245 | 728.7s
step  1000 | train loss 2.3853 | val loss 2.4024 | 949.2s
step  1250 | train loss 2.3624 | val loss 2.3870 | 1168.4s
step  1500 | train loss 2.3287 | val loss 2.3616 | 1388.7s
step  1750 | train loss 2.2865 | val loss 2.3130 | 1607.3s
step  2000 | train loss 2.2234 | val loss 2.2659 | 1821.4s
step  2250 | train loss 2.1720 | val loss 2.2157 | 2033.9s
step  2500 | train loss 2.1206 | val loss 2.1866 | 2260.9s
step  2750 | train loss 2.0825 | val loss 2.1417 | 2474.9s
step  2999 | train loss 2.0497 | val loss 2.1264 | 2683.2s
done in 2683.9s


## 9. Generate Shakespeare-ish text

Greedy decoding from `GPT.generate`, seeded with a newline character.

In [28]:
txt = 'I am Vaibhav'
torch.as_tensor([encode(txt)])

tensor([[21,  1, 39, 51,  1, 34, 39, 47, 40, 46, 39, 60]])

In [45]:
model.eval()
txt='ABA'
context = torch.as_tensor([encode(txt)])
#context = torch.zeros((1, 1), dtype=torch.long, device=device)  # seed token
generated = model.generate(context, max_new=500)
print(len(generated[0].tolist()),len(txt),decode(generated[0].tolist()))

503 3 ABANGENT:
Whe the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the t


## 10. Save the trained weights (optional)

In [11]:
torch.save(model.state_dict(), "gpt_shakespeare.pt")
print("saved to gpt_shakespeare.pt")

saved to gpt_shakespeare.pt
